In [1]:
from __future__ import annotations

import os
import uuid
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "KnowledgeSource"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

sources = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="source_id",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="version",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="status",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: KnowledgeSource


In [5]:
def deterministic_uuid(source_id: str) -> uuid.UUID:
    return uuid.uuid5(uuid.NAMESPACE_URL, source_id)

In [6]:
print(deterministic_uuid("docs/weaviate/vector-search"))
print(deterministic_uuid("docs/weaviate/vector-search"))
print(deterministic_uuid("docs/weaviate/hybrid-search"))

8681bf79-de08-5494-9b53-a536bd4b5541
8681bf79-de08-5494-9b53-a536bd4b5541
3e5ec2c2-b168-5cab-a00d-7d1386003b09


In [7]:
source_data = [
    {
        "source_id": "docs/weaviate/docker-setup",
        "title": "Local Weaviate Docker Setup",
        "content": "A local Weaviate instance can be started in Docker and used for development notebooks and experiments.",
        "topic": "docker",
        "version": 1,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/cloud-connection",
        "title": "Weaviate Cloud Connection",
        "content": "A Weaviate Cloud cluster requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API key.",
        "topic": "cloud",
        "version": 1,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/vector-search",
        "title": "Vector Search",
        "content": "Vector search finds objects by semantic similarity using embeddings generated from text or other data.",
        "topic": "vector_search",
        "version": 1,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/hybrid-search",
        "title": "Hybrid Search",
        "content": "Hybrid search combines BM25 keyword search with vector search and uses alpha to balance both ranking methods.",
        "topic": "hybrid_search",
        "version": 1,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/rag",
        "title": "RAG with Weaviate",
        "content": "RAG retrieves relevant chunks from Weaviate and passes them to a generative model to produce grounded answers.",
        "topic": "rag",
        "version": 1,
        "status": "draft",
    },
]

In [8]:
for item in source_data:
    object_uuid = deterministic_uuid(item["source_id"])

    sources.data.insert(
        properties=item,
        uuid=object_uuid,
    )

    print(item["source_id"], "->", object_uuid)

docs/weaviate/docker-setup -> fc0081f2-6ddb-53d9-b42f-077d00578eb6
docs/weaviate/cloud-connection -> 59536919-adc6-5348-8017-fe99327ba48c
docs/weaviate/vector-search -> 8681bf79-de08-5494-9b53-a536bd4b5541
docs/weaviate/hybrid-search -> 3e5ec2c2-b168-5cab-a00d-7d1386003b09
docs/weaviate/rag -> b59c791a-308d-52fb-b881-b94b65940e66


In [9]:
response = sources.query.fetch_objects(
    limit=10,
    return_properties=[
        "source_id",
        "title",
        "topic",
        "version",
        "status",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print(obj.properties)
    print("-" * 80)

UUID: 3e5ec2c2-b168-5cab-a00d-7d1386003b09
{'source_id': 'docs/weaviate/hybrid-search', 'version': 1, 'topic': 'hybrid_search', 'title': 'Hybrid Search', 'status': 'published'}
--------------------------------------------------------------------------------
UUID: 59536919-adc6-5348-8017-fe99327ba48c
{'source_id': 'docs/weaviate/cloud-connection', 'version': 1, 'topic': 'cloud', 'title': 'Weaviate Cloud Connection', 'status': 'published'}
--------------------------------------------------------------------------------
UUID: 8681bf79-de08-5494-9b53-a536bd4b5541
{'source_id': 'docs/weaviate/vector-search', 'version': 1, 'topic': 'vector_search', 'title': 'Vector Search', 'status': 'published'}
--------------------------------------------------------------------------------
UUID: b59c791a-308d-52fb-b881-b94b65940e66
{'source_id': 'docs/weaviate/rag', 'version': 1, 'topic': 'rag', 'title': 'RAG with Weaviate', 'status': 'draft'}
--------------------------------------------------------------

In [10]:
duplicate_item = source_data[0]
duplicate_uuid = deterministic_uuid(duplicate_item["source_id"])

try:
    sources.data.insert(
        properties=duplicate_item,
        uuid=duplicate_uuid,
    )
except Exception as error:
    print(type(error).__name__)
    print(error)

UnexpectedStatusCodeError
Object was not added! Unexpected status code: 422, with response body: {'error': [{'message': "id 'fc0081f2-6ddb-53d9-b42f-077d00578eb6' already exists"}]}.


In [11]:
def object_exists(object_uuid: uuid.UUID) -> bool:
    obj = sources.query.fetch_object_by_id(
        uuid=object_uuid,
        return_properties=["source_id"],
    )

    return obj is not None

In [12]:
existing_uuid = deterministic_uuid("docs/weaviate/vector-search")
missing_uuid = deterministic_uuid("docs/weaviate/not-existing-document")

print("Existing:", object_exists(existing_uuid))
print("Missing:", object_exists(missing_uuid))

Existing: True
Missing: False


In [13]:
def upsert_source(item: dict[str, object]) -> str:
    source_id = str(item["source_id"])
    object_uuid = deterministic_uuid(source_id)

    if object_exists(object_uuid):
        sources.data.update(
            uuid=object_uuid,
            properties=item,
        )
        return f"Updated: {source_id} -> {object_uuid}"

    sources.data.insert(
        properties=item,
        uuid=object_uuid,
    )
    return f"Inserted: {source_id} -> {object_uuid}"

In [14]:
for item in source_data:
    result = upsert_source(item)
    print(result)

Updated: docs/weaviate/docker-setup -> fc0081f2-6ddb-53d9-b42f-077d00578eb6
Updated: docs/weaviate/cloud-connection -> 59536919-adc6-5348-8017-fe99327ba48c
Updated: docs/weaviate/vector-search -> 8681bf79-de08-5494-9b53-a536bd4b5541
Updated: docs/weaviate/hybrid-search -> 3e5ec2c2-b168-5cab-a00d-7d1386003b09
Updated: docs/weaviate/rag -> b59c791a-308d-52fb-b881-b94b65940e66


In [15]:
updated_source_data = [
    {
        "source_id": "docs/weaviate/rag",
        "title": "RAG with Weaviate",
        "content": (
            "RAG retrieves relevant chunks from Weaviate and passes them to a generative model. "
            "A focused RAG collection should contain clean chunks from notebooks and Markdown notes."
        ),
        "topic": "rag",
        "version": 2,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/named-vectors",
        "title": "Named Vectors",
        "content": (
            "Named vectors allow one object to have multiple vector representations, "
            "for example title_vector and description_vector."
        ),
        "topic": "named_vectors",
        "version": 1,
        "status": "published",
    },
]

In [16]:
for item in updated_source_data:
    result = upsert_source(item)
    print(result)

Updated: docs/weaviate/rag -> b59c791a-308d-52fb-b881-b94b65940e66
Inserted: docs/weaviate/named-vectors -> 16a7926e-b02e-5f04-ae00-c854ea13ea5a


In [17]:
response = sources.query.fetch_objects(
    limit=20,
    return_properties=[
        "source_id",
        "title",
        "content",
        "topic",
        "version",
        "status",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Source ID:", obj.properties["source_id"])
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Version:", obj.properties["version"])
    print("Status:", obj.properties["status"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

UUID: 16a7926e-b02e-5f04-ae00-c854ea13ea5a
Source ID: docs/weaviate/named-vectors
Title: Named Vectors
Topic: named_vectors
Version: 1
Status: published
Content: Named vectors allow one object to have multiple vector representations, for example title_vector and description_vector.
--------------------------------------------------------------------------------
UUID: 3e5ec2c2-b168-5cab-a00d-7d1386003b09
Source ID: docs/weaviate/hybrid-search
Title: Hybrid Search
Topic: hybrid_search
Version: 1
Status: published
Content: Hybrid search combines BM25 keyword search with vector search and uses alpha to balance both ranking methods.
--------------------------------------------------------------------------------
UUID: 59536919-adc6-5348-8017-fe99327ba48c
Source ID: docs/weaviate/cloud-connection
Title: Weaviate Cloud Connection
Topic: cloud
Version: 1
Status: published
Content: A Weaviate Cloud cluster requires a cluster URL, a Weaviate API key and provider headers such as the OpenAI API ke

In [18]:
response = sources.query.fetch_objects(
    filters=Filter.by_property("status").equal("published"),
    limit=20,
    return_properties=[
        "source_id",
        "title",
        "topic",
        "version",
        "status",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'source_id': 'docs/weaviate/docker-setup', 'version': 1, 'topic': 'docker', 'title': 'Local Weaviate Docker Setup', 'status': 'published'}
{'source_id': 'docs/weaviate/cloud-connection', 'version': 1, 'topic': 'cloud', 'title': 'Weaviate Cloud Connection', 'status': 'published'}
{'source_id': 'docs/weaviate/vector-search', 'version': 1, 'topic': 'vector_search', 'title': 'Vector Search', 'status': 'published'}
{'source_id': 'docs/weaviate/hybrid-search', 'version': 1, 'topic': 'hybrid_search', 'title': 'Hybrid Search', 'status': 'published'}
{'source_id': 'docs/weaviate/rag', 'version': 2, 'topic': 'rag', 'title': 'RAG with Weaviate', 'status': 'published'}
{'source_id': 'docs/weaviate/named-vectors', 'version': 1, 'topic': 'named_vectors', 'title': 'Named Vectors', 'status': 'published'}


In [19]:
response = sources.query.near_text(
    query="how to build RAG using notebooks and markdown notes",
    filters=Filter.by_property("status").equal("published"),
    limit=5,
    return_properties=[
        "source_id",
        "title",
        "content",
        "topic",
        "version",
        "status",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Source ID:", obj.properties["source_id"])
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Version:", obj.properties["version"])
    print("Status:", obj.properties["status"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.3361290693283081
Source ID: docs/weaviate/rag
Title: RAG with Weaviate
Topic: rag
Version: 2
Status: published
Content: RAG retrieves relevant chunks from Weaviate and passes them to a generative model. A focused RAG collection should contain clean chunks from notebooks and Markdown notes.
--------------------------------------------------------------------------------
Distance: 0.6361950039863586
Source ID: docs/weaviate/docker-setup
Title: Local Weaviate Docker Setup
Topic: docker
Version: 1
Status: published
Content: A local Weaviate instance can be started in Docker and used for development notebooks and experiments.
--------------------------------------------------------------------------------
Distance: 0.7562865614891052
Source ID: docs/weaviate/named-vectors
Title: Named Vectors
Topic: named_vectors
Version: 1
Status: published
Content: Named vectors allow one object to have multiple vector representations, for example title_vector and description_vector.
---------

In [20]:
response = sources.generate.near_text(
    query="explain RAG and named vectors in Weaviate",
    filters=Filter.by_property("status").equal("published"),
    grouped_task=(
        "Answer using only the retrieved knowledge source objects. "
        "Explain the concepts briefly and mention the source titles."
    ),
    limit=5,
    return_properties=[
        "source_id",
        "title",
        "content",
        "topic",
        "version",
        "status",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "|",
        obj.properties["source_id"],
        "| version:",
        obj.properties["version"],
    )

Here are brief explanations of the relevant concepts along with their source titles:

1. **RAG (Retrieval-Augmented Generation)**: This technique retrieves relevant chunks of data from a source (e.g., Weaviate) and feeds them into a generative model to improve the quality of responses. It is important for the RAG collection to include clean chunks from sources like notebooks and Markdown notes. 
   - Source: **RAG with Weaviate**

2. **Named Vectors**: This concept allows a single object in a database to have multiple vector representations. For instance, an object could be represented by different vectors for its title and description, allowing for more nuanced similarity searches.
   - Source: **Named Vectors**

3. **Local Weaviate Instance**: You can set up a local instance of Weaviate using Docker, which is useful for development, testing, and experimentation with notebooks.
   - Source: **Local Weaviate Docker Setup**

4. **Vector Search**: This method identifies objects based on 

In [21]:
current_source_ids = {
    item["source_id"]
    for item in source_data + updated_source_data
}

response = sources.query.fetch_objects(
    limit=100,
    return_properties=[
        "source_id",
        "title",
        "status",
    ],
)

existing_source_ids = {
    obj.properties["source_id"]
    for obj in response.objects
}

missing_in_source = existing_source_ids - current_source_ids

print("Existing in Weaviate:", existing_source_ids)
print("Current source IDs:", current_source_ids)
print("Missing in source:", missing_in_source)

Existing in Weaviate: {'docs/weaviate/vector-search', 'docs/weaviate/hybrid-search', 'docs/weaviate/named-vectors', 'docs/weaviate/cloud-connection', 'docs/weaviate/rag', 'docs/weaviate/docker-setup'}
Current source IDs: {'docs/weaviate/vector-search', 'docs/weaviate/hybrid-search', 'docs/weaviate/named-vectors', 'docs/weaviate/cloud-connection', 'docs/weaviate/rag', 'docs/weaviate/docker-setup'}
Missing in source: set()


In [22]:
for source_id in missing_in_source:
    object_uuid = deterministic_uuid(source_id)

    sources.data.update(
        uuid=object_uuid,
        properties={
            "status": "archived",
        },
    )

    print("Archived:", source_id)

In [23]:
def sync_sources(items: list[dict[str, object]]) -> None:
    current_source_ids = {
        str(item["source_id"])
        for item in items
    }

    for item in items:
        print(upsert_source(item))

    response = sources.query.fetch_objects(
        limit=100,
        return_properties=["source_id"],
    )

    existing_source_ids = {
        obj.properties["source_id"]
        for obj in response.objects
    }

    missing_source_ids = existing_source_ids - current_source_ids

    for source_id in missing_source_ids:
        object_uuid = deterministic_uuid(source_id)

        sources.data.update(
            uuid=object_uuid,
            properties={
                "status": "archived",
            },
        )

        print("Archived:", source_id)

In [24]:
latest_source_data = [
    {
        "source_id": "docs/weaviate/cloud-connection",
        "title": "Weaviate Cloud Connection",
        "content": "Connect to Weaviate Cloud with a cluster URL, Weaviate API key and OpenAI provider header.",
        "topic": "cloud",
        "version": 2,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/rag",
        "title": "RAG with Weaviate",
        "content": "Focused RAG works best when the collection contains clean notebook chunks and Markdown concept notes.",
        "topic": "rag",
        "version": 3,
        "status": "published",
    },
    {
        "source_id": "docs/weaviate/multi-tenancy",
        "title": "Multi-Tenancy",
        "content": "Multi-tenancy isolates data for different customers or organizations inside one shared collection schema.",
        "topic": "multi_tenancy",
        "version": 1,
        "status": "published",
    },
]

sync_sources(latest_source_data)

Updated: docs/weaviate/cloud-connection -> 59536919-adc6-5348-8017-fe99327ba48c
Updated: docs/weaviate/rag -> b59c791a-308d-52fb-b881-b94b65940e66
Inserted: docs/weaviate/multi-tenancy -> e90ee0b2-98cf-57bb-9428-9cd52c1f6d46
Archived: docs/weaviate/vector-search
Archived: docs/weaviate/named-vectors
Archived: docs/weaviate/docker-setup
Archived: docs/weaviate/hybrid-search


In [25]:
response = sources.query.fetch_objects(
    limit=100,
    return_properties=[
        "source_id",
        "title",
        "topic",
        "version",
        "status",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'source_id': 'docs/weaviate/named-vectors', 'version': 1, 'topic': 'named_vectors', 'title': 'Named Vectors', 'status': 'archived'}
{'source_id': 'docs/weaviate/hybrid-search', 'version': 1, 'topic': 'hybrid_search', 'title': 'Hybrid Search', 'status': 'archived'}
{'source_id': 'docs/weaviate/cloud-connection', 'version': 2, 'topic': 'cloud', 'title': 'Weaviate Cloud Connection', 'status': 'published'}
{'source_id': 'docs/weaviate/vector-search', 'version': 1, 'topic': 'vector_search', 'title': 'Vector Search', 'status': 'archived'}
{'source_id': 'docs/weaviate/rag', 'version': 3, 'topic': 'rag', 'title': 'RAG with Weaviate', 'status': 'published'}
{'source_id': 'docs/weaviate/multi-tenancy', 'version': 1, 'topic': 'multi_tenancy', 'title': 'Multi-Tenancy', 'status': 'published'}
{'source_id': 'docs/weaviate/docker-setup', 'version': 1, 'topic': 'docker', 'title': 'Local Weaviate Docker Setup', 'status': 'archived'}


In [26]:
response = sources.query.near_text(
    query="how to connect to Weaviate Cloud and use RAG",
    filters=Filter.by_property("status").equal("published"),
    limit=5,
    return_properties=[
        "source_id",
        "title",
        "content",
        "topic",
        "version",
        "status",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Status:", obj.properties["status"])
    print("Version:", obj.properties["version"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.3806096911430359
Title: Weaviate Cloud Connection
Status: published
Version: 2
Content: Connect to Weaviate Cloud with a cluster URL, Weaviate API key and OpenAI provider header.
--------------------------------------------------------------------------------
Distance: 0.43654197454452515
Title: RAG with Weaviate
Status: published
Version: 3
Content: Focused RAG works best when the collection contains clean notebook chunks and Markdown concept notes.
--------------------------------------------------------------------------------
Distance: 0.6905539631843567
Title: Multi-Tenancy
Status: published
Version: 1
Content: Multi-tenancy isolates data for different customers or organizations inside one shared collection schema.
--------------------------------------------------------------------------------


In [27]:
client.close()